In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
dataset_path = "../Dataset"

img_size = (224,224)
batch_size = 32

In [3]:
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    shear_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

In [4]:
train_data = datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

Found 3601 images belonging to 3 classes.


In [5]:
val_data = datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

Found 899 images belonging to 3 classes.


In [6]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [7]:
base_model.trainable = False

In [8]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(train_data.num_classes, activation='softmax')
])

In [9]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [10]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [11]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=[early_stop]
)

Epoch 1/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 223s 2s/step - accuracy: 0.8728 - loss: 0.2999 - val_accuracy: 0.9266 - val_loss: 0.1839
Epoch 2/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 344s 3s/step - accuracy: 0.9372 - loss: 0.1630 - val_accuracy: 0.9288 - val_loss: 0.1797
Epoch 3/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 213s 2s/step - accuracy: 0.9433 - loss: 0.1404 - val_accuracy: 0.9422 - val_loss: 0.1423
Epoch 4/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 197s 2s/step - accuracy: 0.9547 - loss: 0.1276 - val_accuracy: 0.9455 - val_loss: 0.1533
Epoch 5/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 199s 2s/step - accuracy: 0.9542 - loss: 0.1240 - val_accuracy: 0.9422 - val_loss: 0.1495
Epoch 6/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 202s 2s/step - accuracy: 0.9625 - loss: 0.1062 - val_accuracy: 0.9299 - val_loss: 0.1783


In [12]:
model.save("../model/plant_disease_model.h5")

In [13]:
print(train_data.class_indices)

{'Early_Blight': 0, 'Healthy': 1, 'Late_Blight': 2}
